# 11 - Mean relative difference

Bringing it all together. Let's see how we do.

In [1]:
from nested_pandas.utils import count_nested
import numpy as np
import pandas as pd
import nested_pandas as npd
import numpy.ma as ma
import math
from hats import HealpixPixel
from homework_plotting import per_detector
from tqdm import tqdm
import hats
from hats.catalog import PartitionInfo
import lsdb
from lsdb import ConeSearch, PixelSearch
from nested_pandas.utils import count_nested
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dask.distributed import print as dask_print
from dask.distributed import Client
from nested_pandas.utils import count_nested
from dask.distributed import print as dask_print
import dask
from lsdb.streams import CatalogStream
from tqdm import tqdm

lsdb.show_versions()

# lsdb.show_versions()


--------      SYSTEM INFO      --------
python        : 3.12.11
python-bits   : 64
OS            : Linux
OS-release    : 5.14.0-570.58.1.el9_6.x86_64
Version       : #1 SMP PREEMPT_DYNAMIC Fri Oct 31 13:55:05 UTC 2025
machine       : x86_64
processor     : 
byteorder     : little
LC_ALL        : 
LANG          : 
--------   INSTALLED VERSIONS   --------
lsdb          : 0.8.2
hats          : 0.8.2
nested-pandas : 0.6.8
pandas        : 2.3.1
numpy         : 2.4.2
dask          : 2025.7.0
pyarrow       : 17.0.0
fsspec        : 2025.7.0


In [2]:
object_columns = ['coord_dec', 'coord_ra', 
       'objectId', 
	'g_psfFlux',       'g_psfFluxErr', 
       'i_psfFlux', 'i_psfFluxErr',
       'r_psfFlux',       'r_psfFluxErr', 
       'u_psfFlux',       'u_psfFluxErr', 
       'y_psfFlux', 'y_psfFluxErr',
       'z_psfFlux', 'z_psfFluxErr',
       'objectForcedSource.band',
     'objectForcedSource.detector',
     'objectForcedSource.invalidPsfFlag',
     'objectForcedSource.midpointMjdTai',
     'objectForcedSource.pixelFlags_bad',
     'objectForcedSource.pixelFlags_cr',
     'objectForcedSource.pixelFlags_crCenter',
     'objectForcedSource.pixelFlags_edge',
     'objectForcedSource.pixelFlags_interpolated',
     'objectForcedSource.pixelFlags_interpolatedCenter',
     'objectForcedSource.pixelFlags_nodata',
     'objectForcedSource.pixelFlags_saturated',
     'objectForcedSource.pixelFlags_saturatedCenter',
     'objectForcedSource.pixelFlags_suspect',
     'objectForcedSource.pixelFlags_suspectCenter',
     'objectForcedSource.psfDiffFlux',
     'objectForcedSource.psfDiffFlux_flag',
     'objectForcedSource.psfDiffFluxErr',
     'objectForcedSource.psfFlux',
     'objectForcedSource.psfFlux_flag',
     'objectForcedSource.psfFluxErr']

In [3]:
objectForcedSource_flag_columns = ['objectForcedSource.invalidPsfFlag',
 'objectForcedSource.pixelFlags_bad',
 'objectForcedSource.pixelFlags_cr',
 'objectForcedSource.pixelFlags_crCenter',
 'objectForcedSource.pixelFlags_edge',
 'objectForcedSource.pixelFlags_interpolated',
 'objectForcedSource.pixelFlags_interpolatedCenter',
 'objectForcedSource.pixelFlags_nodata',
 'objectForcedSource.pixelFlags_saturated',
 'objectForcedSource.pixelFlags_saturatedCenter',
 'objectForcedSource.pixelFlags_suspect',
 'objectForcedSource.pixelFlags_suspectCenter',
 'objectForcedSource.psfDiffFlux_flag',
 'objectForcedSource.psfFlux_flag']

flag_query = " and ".join(objectForcedSource_flag_columns)
count_query = " or ".join([f"n_lc_{band} >= 10" for band in 'ugrizy'])

In [4]:
def limit_lightcurves(frame):
    # Count by band
    counts = count_nested(frame, f"objectForcedSource", by="band", join=False)

    for band in 'ugrizy':
        
        if f"n_objectForcedSource_{band}" in counts.columns:
            # Drop light curves with less than 10 points
            frame[f"lc_{band}"] = frame[counts[f"n_objectForcedSource_{band}"] > 10].query(f"objectForcedSource.band=='{band}'")["objectForcedSource"]
            frame = count_nested(frame, f"lc_{band}", join=True)
    
        else:
            # there are no light curves - create an empty frame
            frame[f"lc_{band}"] = frame.query(f"objectForcedSource.band=='{band}'")["objectForcedSource"]
            frame = count_nested(frame, f"lc_{band}", join=True)

    # # Drop objects that have NO light curves long enough
    frame = frame.query(count_query)


    # We don't need the flag colums anymore
    for col in objectForcedSource_flag_columns:
        frame = frame.drop(col, axis=1)

    # We also don't need the full list of sources - they're split to single-band light curves
    frame = frame.drop("objectForcedSource", axis=1)
    return frame
    

def get_relative_difference(row):
    results = {}
    for band in "ugrizy":
        if pd.isna(row[f"{band}_psfFluxErr"]):
            results |= {               f"lc_{band}.rel_diff":np.zeros(len(row[f"lc_{band}.psfFlux"]), dtype=np.float64)}
            # pass
        elif np.isnan(row[f"lc_{band}.psfFluxErr"]).any():
            results |=  {               f"lc_{band}.rel_diff":np.zeros(len(row[f"lc_{band}.psfFlux"]), dtype=np.float64)}
            # pass
        else:
            results |=  {f"lc_{band}.rel_diff":np.array((row[f"lc_{band}.psfFlux"] - row[f"{band}_psfFlux"]) / np.sqrt(row[f"lc_{band}.psfFluxErr"]**2 + row[f"{band}_psfFluxErr"]**2), dtype=np.float64)}
    return results

def relative_differences(rel_diffs, pixel):
    summaries = {"Norder": pixel.order, "Npix": pixel.pixel}
    
    for band in "ugrizy":
        flat = rel_diffs[f"lc_{band}"].nest.to_flat().reset_index(drop=True)

        sums = np.zeros(189, dtype=np.float64)
        sum_squares = np.zeros(189, dtype=np.float64)
        
        sums[flat["detector"]] += flat["rel_diff"]
        sum_squares[flat["detector"]] += flat["rel_diff"]**2
        counts = np.histogram(flat["detector"], bins=np.arange(190))[0]
        summaries[f"diffs_{band}.sums"] = [sums]
        summaries[f"diffs_{band}.sum_squares"] = [sum_squares]
        summaries[f"diffs_{band}.counts"] = [counts]

    nested_thing = npd.NestedFrame(summaries)
    return nested_thing

In [5]:
non_empty_partitions = PartitionInfo._read_from_csv("non_empty_partitions.csv")

In [6]:
client = Client(n_workers=2, threads_per_worker=1, memory_limit="8GB") 

In [7]:
ocat = lsdb.open_catalog('/sdf/data/rubin/shared/lsdb_commissioning/hats/v30_0_0_rc2/object_collection', columns=object_columns, search_filter=PixelSearch(non_empty_partitions[0:1000]))
ocat

ERROR:tornado.application:Uncaught exception GET /status/ws (127.0.0.1)
HTTPServerRequest(protocol='http', host='usdf-rsp.slac.stanford.edu', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "/opt/lsst/software/stack/conda/envs/lsst-scipipe-10.1.0-exact/lib/python3.12/site-packages/tornado/websocket.py", line 965, in _accept_connection
    open_result = handler.open(*handler.open_args, **handler.open_kwargs)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/lsst/software/stack/conda/envs/lsst-scipipe-10.1.0-exact/lib/python3.12/site-packages/tornado/web.py", line 3375, in wrapper
    return method(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/lsst/software/stack/conda/envs/lsst-scipipe-10.1.0-exact/lib/python3.12/site-packages/bokeh/server/views/ws.py", line 149, in open
    raise ProtocolError("Token is expired. Configure the app with a larger value f

,coord_dec,coord_ra,objectId,g_psfFlux,g_psfFluxErr,i_psfFlux,i_psfFluxErr,r_psfFlux,r_psfFluxErr,u_psfFlux,u_psfFluxErr,y_psfFlux,y_psfFluxErr,z_psfFlux,z_psfFluxErr,objectForcedSource
npartitions=1000,,,,,,,,,,,,,,,,
"Order: 8, Pixel: 273056",double[pyarrow],double[pyarrow],int64[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],"nested<band: [string], detector: [int16], inva..."
"Order: 8, Pixel: 273058",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Order: 9, Pixel: 1744015",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Order: 8, Pixel: 436004",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...


In [8]:
my_meta = {"Norder": int, "Npix": int}
diff_meta = {}
for band in "ugrizy":
    my_meta[f"diffs_{band}.sums"] = np.float64
    my_meta[f"diffs_{band}.sum_squares"] = np.float64
    my_meta[f"diffs_{band}.counts"] = np.int64

    # diff_meta[f"diff_{band}"] = np.int64
    diff_meta[f"lc_{band}.rel_diff"] = np.float64

In [9]:
all_results = ocat.map_partitions(limit_lightcurves).map_rows(get_relative_difference, meta=diff_meta, infer_nesting=True, append_columns=True).map_partitions(relative_differences, meta=my_meta,include_pixel=True).compute()
# all_results = ocat.map_partitions(limit_lightcurves).map_rows(get_relative_difference, meta=diff_meta).map_partitions(relative_differences, meta=my_meta,include_pixel=True).compute()
all_results.to_parquet("full_results/part0.parquet")